Sistema de clasificación de imágenes

El conjunto de datos se compone de fotos de perros y gatos proporcionadas como un subconjunto de fotos de uno mucho más grande de 3 millones de fotos anotadas manualmente. Estos datos se obtuvieron a través de una colaboración entre Petfinder.com y Microsoft.

El conjunto de datos se usó originalmente como un CAPTCHA, es decir, una tarea que se cree que un humano encuentra trivial, pero que una máquina no puede resolver, que se usa en sitios web para distinguir entre usuarios humanos y bots. La tarea se denominó "Asirra". Cuando se presentó "Asirra", se mencionó "que los estudios de usuarios indican que los humanos pueden resolverlo el 99,6% de las veces en menos de 30 segundos". A menos que se produzca un gran avance en la visión artificial, esperamos que los ordenadores no tengan más de 1/54.000 posibilidades de resolverlo.

En el momento en que se publicó la competencia, el resultado de última generación se logró con un SVM y se describió en un artículo de 2007 con el título "Ataques de Machine Learning contra el CAPTCHA de Asirra" (PDF) que logró una precisión de clasificación del 80%. Fue este documento el que demostró que la tarea ya no era una tarea adecuada para un CAPTCHA poco después de que se propusiera la tarea.


Paso 1: Carga del conjunto de datos

El conjunto de datos se encuentra en este link. Descarga la carpeta y descomprime los archivos. Ahora tendrás una carpeta con el dataset y una carpeta llamada train que contiene más de 25.000 archivos de imagen (formato .jpg) de perros y gatos. Las fotos están etiquetadas por su nombre de archivo, con la palabra dog o cat.

Paso 2: Visualiza la información de entrada

El primer paso cuando nos enfrentamos a un problema de clasificación de imágenes es obtener toda la información posible a través de ellas. Por lo tanto, carga e imprime las primeras nueve fotos de perros en una sola figura. Repite lo mismo para los gatos. Puedes ver que las fotos son a color y tienen diferentes formas y tamaños.

Esta variedad de tamaños y formatos debe solucionarse antes de entrenar el modelo. Asegúrate de que todas tengan un tamaño fijo de 200x200 píxeles.

Como podrás ver, son una gran cantidad de imágenes, asegúrate de seguir las siguientes normas:

Si tienes más de 12 gigabytes de RAM, usa la API de procesamiento de imágenes de Keras para cargar las 25.000 fotos en el conjunto de datos de entrenamiento y remodelarlas a fotos cuadradas de 200×200 píxeles. La etiqueta también debe determinarse para cada foto en función de los nombres de archivo. Se debe guardar una tupla de fotos y etiquetas.
Si no tienes más de 12 gigabytes de RAM, carga las imágenes progresivamente usando la clase Keras ImageDataGenerator y la función flow_from_directory(). Esto será más lento de ejecutar, pero se ejecutará en hardware de menor capacidad. Esta función prefiere que los datos se dividan en directorios train y test separados, y debajo de cada directorio para tener un subdirectorio para cada clase.
Una vez tengas todas las imágenes procesadas, crea un objeto ImageDataGenerator para datos de entrenamiento y prueba. Luego pasa la carpeta que tiene datos de entrenamiento al objeto trdata y, de manera similar, pasa la carpeta que tiene datos de prueba al objeto tsdata. De esta forma, se etiquetarán las imágenes automáticamente y estará todo listo para entrar a la red.

Paso 3: Construye una RNA

Cualquier clasificador que se ajuste a este problema tendrá que ser robusto porque algunas imágenes muestran al gato o al perro en una esquina o tal vez a 2 gatos o perros en la misma foto. Si has podido investigar algunas de las implementaciones de los ganadores de otras competiciones también relacionadas con imágenes, verás que VGG16 es una arquitectura de CNN utilizada para ganar la competencia de Kaggle ILSVR (Imagenet) en 2014. Se considera una de las arquitecturas de modelos de visión con mejores resultados hasta la fecha.

Utiliza la siguiente arquitectura de prueba:

model = Sequential()

model.add(Conv2D(input_shape = (224,224,3), filters = 64, kernel_size = (3,3), padding = "same", activation = "relu"))

model.add(Conv2D(filters = 64,kernel_size = (3,3),padding = "same", activation = "relu"))

model.add(MaxPool2D(pool_size = (2,2),strides = (2,2)))

model.add(Conv2D(filters = 128, kernel_size = (3,3), padding = "same", activation = "relu"))

model.add(Conv2D(filters = 128, kernel_size = (3,3), padding = "same", activation = "relu"))

model.add(MaxPool2D(pool_size = (2,2),strides = (2,2)))

model.add(Conv2D(filters = 256, kernel_size = (3,3), padding = "same", activation = "relu"))

model.add(Conv2D(filters = 256, kernel_size = (3,3), padding = "same", activation = "relu"))

model.add(Conv2D(filters = 256, kernel_size = (3,3), padding = "same", activation = "relu"))

model.add(MaxPool2D(pool_size = (2,2),strides = (2,2)))

model.add(Conv2D(filters = 512, kernel_size = (3,3), padding = "same", activation = "relu"))

model.add(Conv2D(filters = 512, kernel_size = (3,3), padding = "same", activation = "relu"))

model.add(Conv2D(filters = 512, kernel_size = (3,3), padding = "same", activation = "relu"))

model.add(MaxPool2D(pool_size = (2,2),strides = (2,2)))

model.add(Conv2D(filters = 512, kernel_size = (3,3), padding = "same", activation = "relu"))

model.add(Conv2D(filters = 512, kernel_size = (3,3), padding = "same", activation = "relu"))

model.add(Conv2D(filters = 512, kernel_size = (3,3), padding = "same", activation = "relu"))

model.add(MaxPool2D(pool_size = (2,2),strides = (2,2)))

model.add(Flatten())

model.add(Dense(units = 4096,activation = "relu"))

model.add(Dense(units = 4096,activation = "relu"))

model.add(Dense(units = 2, activation = "softmax"))

El código anterior aplica convoluciones a los datos (capas Conv2D y MaxPool2D) y después aplica capas densas (capas Dense) para el procesamiento de los valores numéricos obtenidos tras las convoluciones.

A continuación añade los elementos restantes para conformar el modelo, entrénalo y mide su rendimiento.

Paso 4: Optimiza el modelo anterior

Importa el método ModelCheckpoint y EarlyStopping de Keras. Crea un objeto de ambos y pásalo como funciones callback a fit_generator.

Carga el mejor modelo de los anteriores y utiliza el conjunto de test para hacer predicciones.

Paso 5: Guarda el modelo

Almacena el modelo en la carpeta correspondiente.

In [ ]:
import os
import shutil
import random
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPool2D, Flatten, Dense
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image

In [ ]:
# 1. Ver el directorio actual de trabajo
print(f"Directorio actual: {os.getcwd()}")
F
# 2. Definir la ruta que intentamos buscar
# Nota: Usamos os.path.join para que funcione en cualquier sistema operativo
ruta_buscada = os.path.join('natbp_intro_ML', 'data', 'raw', 'dogs-vs-cats', 'train')

print(f"Buscando la ruta: {ruta_buscada}")

if os.path.exists(ruta_buscada):
    print("✅ ¡Ruta encontrada con éxito!")
else:
    print("❌ No se encontró la ruta. Listando carpetas en el directorio actual para ayudar:")
    print(os.listdir('.'))

    # Intento de búsqueda recursiva simple si no se encuentra
    print("\nIntentando buscar la carpeta 'dogs-vs-cats' en los alrededores...")
    encontrado = False
    for root, dirs, files in os.walk('.'):
        if 'dogs-vs-cats' in dirs:
            print(f"💡 Sugerencia: Se encontró la carpeta en: {os.path.abspath(os.path.join(root, 'dogs-vs-cats'))}")
            encontrado = True
            break
    if not encontrado:
        print("No se encontró la carpeta 'dogs-vs-cats'. Por favor, verifica que el notebook esté en la misma carpeta raíz que 'natbp_intro_ML'.")

Directorio actual: c:\Users\nata1\Desktop\4geeks\intro-ML\natbp_intro_ML\src
Buscando la ruta: natbp_intro_ML\data\raw\dogs-vs-cats\train
❌ No se encontró la ruta. Listando carpetas en el directorio actual para ayudar:
['00-eda-airbnb-ny.ipynb', '01-logistic-regression-bank-marketing-campaign-data.ipynb', '02-linear-regression-medical-insurance-cost.ipynb', '03-linear-regression-reg-demographic-health-data.ipynb', '04-decision-tree-diabetes.ipynb', '04-eda-diabetes.ipynb', '05-random-forest-diabetes.ipynb', '06-boosting-diabetes.ipynb', '07-naive-bayes-playstore-reviews.ipynb', '08-k-nearest-neighbours-winequality-red.ipynb', '09-k-means-housing.ipynb', '10-time-series-forecasting-sales.ipynb', '11-image-classifier-dogs-vs-cats.ipynb', '12-nlp-url-spam.ipynb', '13-recommendation-systems-adult-census-income.ipynb', 'app.py', 'dataset_procesado', 'explore.ipynb', 'utils.py']

Intentando buscar la carpeta 'dogs-vs-cats' en los alrededores...
No se encontró la carpeta 'dogs-vs-cats'. Por f

In [ ]:
raw_data_path = os.path.join('..', 'data', 'raw', 'dogs-vs-cats', 'train')
base_dir = os.path.join('..', 'data', 'processed')
train_dir = os.path.join(base_dir, 'train')
test_dir = os.path.join(base_dir, 'test')

# Crear estructura de carpetas para Keras
for root in [train_dir, test_dir]:
    for label in ['dog', 'cat']:
        os.makedirs(os.path.join(root, label), exist_ok=True)

print(f"✅ Carpeta de origen: {os.path.abspath(raw_data_path)}")
print(f"✅ Carpeta de destino: {os.path.abspath(base_dir)}")

def organize_dataset(src_path, train_dest, test_dest, split=0.2):
    """
    Organiza las imágenes cat.N.jpg y dog.N.jpg en subcarpetas.
    """
    if not os.path.exists(src_path):
        print(f"Error: No se encontró la ruta {src_path}")
        print("Asegúrate de que la ruta sea correcta en tu sistema local.")
        return

    files = [f for f in os.listdir(src_path) if f.endswith('.jpg')]
    random.shuffle(files)

    split_idx = int(len(files) * (1 - split))
    train_files = files[:split_idx]
    test_files = files[split_idx:]

    print(f"Procesando {len(files)} imágenes...")

    for f in train_files:
        # Identificar clase por el nombre del archivo (cat.0.jpg -> cat)
        label = 'cat' if f.lower().startswith('cat') else 'dog'
        shutil.copy(os.path.join(src_path, f), os.path.join(train_dest, label, f))

    for f in test_files:
        label = 'cat' if f.lower().startswith('cat') else 'dog'
        shutil.copy(os.path.join(src_path, f), os.path.join(test_dest, label, f))

    print("Organización completada con éxito.")


organize_dataset(raw_data_path, train_dir, test_dir)

Error: No se encontró la ruta natbp_intro_ML/data/raw/dogs-vs-cats/train
Asegúrate de que la ruta sea correcta en tu sistema local.


In [ ]:
def plot_images(directory, label, n=9):
    path = os.path.join(directory, label)
    if not os.path.exists(path):
        print(f"Directorio no encontrado: {path}")
        return

    files = [f for f in os.listdir(path) if f.endswith('.jpg')]

    plt.figure(figsize=(12, 12))
    for i in range(min(n, len(files))):
        plt.subplot(3, 3, i + 1)
        img = mpimg.imread(os.path.join(path, files[i]))
        plt.imshow(img)
        plt.title(f"{label.capitalize()} - {files[i]}")
        plt.axis('off')
    plt.tight_layout()
    plt.show()

plot_images(train_dir, 'dog')
plot_images(train_dir, 'cat')

In [ ]:
IMG_SIZE = (200, 200)
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)
test_datagen = ImageDataGenerator(rescale=1./255)

# Generador de Entrenamiento
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

# Generador de Validación
val_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)

# Generador de Prueba
test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

In [ ]:
model = Sequential()

# Bloque 1
model.add(Conv2D(input_shape=(200, 200, 3), filters=64, kernel_size=(3,3), padding="same", activation="relu"))
model.add(Conv2D(filters=64, kernel_size=(3,3), padding="same", activation="relu"))
model.add(MaxPool2D(pool_size=(2,2), strides=(2,2)))

# Bloque 2
model.add(Conv2D(filters=128, kernel_size=(3,3), padding="same", activation="relu"))
model.add(Conv2D(filters=128, kernel_size=(3,3), padding="same", activation="relu"))
model.add(MaxPool2D(pool_size=(2,2), strides=(2,2)))

# Bloque 3
model.add(Conv2D(filters=256, kernel_size=(3,3), padding="same", activation="relu"))
model.add(Conv2D(filters=256, kernel_size=(3,3), padding="same", activation="relu"))
model.add(Conv2D(filters=256, kernel_size=(3,3), padding="same", activation="relu"))
model.add(MaxPool2D(pool_size=(2,2), strides=(2,2)))

# Bloque 4
model.add(Conv2D(filters=512, kernel_size=(3,3), padding="same", activation="relu"))
model.add(Conv2D(filters=512, kernel_size=(3,3), padding="same", activation="relu"))
model.add(Conv2D(filters=512, kernel_size=(3,3), padding="same", activation="relu"))
model.add(MaxPool2D(pool_size=(2,2), strides=(2,2)))

# Bloque 5
model.add(Conv2D(filters=512, kernel_size=(3,3), padding="same", activation="relu"))
model.add(Conv2D(filters=512, kernel_size=(3,3), padding="same", activation="relu"))
model.add(Conv2D(filters=512, kernel_size=(3,3), padding="same", activation="relu"))
model.add(MaxPool2D(pool_size=(2,2), strides=(2,2)))

# Capas Densas
model.add(Flatten())
model.add(Dense(units=4096, activation="relu"))
model.add(Dense(units=4096, activation="relu"))
model.add(Dense(units=2, activation="softmax"))

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
checkpoint = ModelCheckpoint("mejor_modelo_vgg16.h5", monitor='val_accuracy', verbose=1, save_best_only=True, mode='max')
early_stop = EarlyStopping(monitor='val_accuracy', patience=5, verbose=1, mode='max')

# Entrenamiento
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=20,
    callbacks=[checkpoint, early_stop]
)

In [ ]:
def predict_pet(img_path, model):
    img = image.load_img(img_path, target_size=IMG_SIZE)
    img_array = image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    prediction = model.predict(img_array)
    res = "Perro" if np.argmax(prediction) == 1 else "Gato"

    plt.imshow(img)
    plt.title(f"Resultado: {res}")
    plt.axis('off')
    plt.show()
    return res

In [ ]:
model.save("modelo_final_perros_gatos.h5")
print("Modelo guardado correctamente.")